In [1]:
# imports e configs
import os, random
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from segmentacao import segmentar_grao, desenhar_contorno

# Pasta raiz com imagens (todas as classes em subpastas dentro de `archive`)
DATA_DIR = Path('archive')
# lista ordenada de subpastas (uma por classe)
class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()]) if DATA_DIR.exists() else []
if len(class_dirs) >= 2:
    dark_dir = class_dirs[0]
    medium_dir = class_dirs[1]
else:
    # fallback: se não houver subpastas, utiliza a própria pasta (pode precisar ajustar)
    dark_dir = DATA_DIR
    medium_dir = DATA_DIR

plt.rcParams['figure.dpi'] = 120
random.seed(42)
np.random.seed(42)


ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Funções auxiliares para EDA (nomes em português)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def carregar_features_csv(caminho='outputs/features.csv'):
    df = pd.read_csv(caminho)
    return df


def plot_boxplots_por_rotulo(df, features, caminho_saida='outputs/boxplots_custom.png'):
    fig, axes = plt.subplots(1, len(features), figsize=(4*len(features), 4))
    if len(features) == 1:
        axes = [axes]
    for i, f in enumerate(features):
        try:
            df.boxplot(column=f, by='rotulo', ax=axes[i])
            axes[i].set_title(f)
        except Exception as e:
            axes[i].text(0.5, 0.5, f'Erro em {f}: {e}', ha='center')
    plt.suptitle('')
    plt.tight_layout()
    plt.savefig(caminho_saida)
    plt.close()


def plot_pca_2d(df, caminho_saida='outputs/pca_2d.png'):
    X = df.drop(columns=['rotulo','caminho']).values
    X = np.nan_to_num(X)
    y = df['rotulo'].values
    pca = PCA(n_components=2, random_state=42)
    Z = pca.fit_transform(X)
    plt.figure(figsize=(7,6))
    for rotulo in np.unique(y):
        sel = y == rotulo
        plt.scatter(Z[sel,0], Z[sel,1], label=rotulo, alpha=0.7)
    plt.legend()
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('PCA 2D – projeção')
    plt.tight_layout()
    plt.savefig(caminho_saida)
    plt.close()


def selectkbest_features(df, k=10):
    from sklearn.feature_selection import SelectKBest, f_classif
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    skb = SelectKBest(f_classif, k=min(k, X.shape[1]))
    skb.fit(X, y)
    nomes = [f for f, s in zip(df.drop(columns=['rotulo','caminho']).columns, skb.get_support()) if s]
    return nomes


def importancia_random_forest(df):
    from sklearn.ensemble import RandomForestClassifier
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    importancias = rf.feature_importances_
    nomes = df.drop(columns=['rotulo','caminho']).columns
    return pd.Series(importancias, index=nomes).sort_values(ascending=False)


def coeficientes_regressao_logistica(df):
    from sklearn.linear_model import LogisticRegression
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X, y)
    nomes = df.drop(columns=['rotulo','caminho']).columns
    coefs = np.mean(np.abs(lr.coef_), axis=0) if lr.coef_.ndim > 1 else lr.coef_
    return pd.Series(coefs, index=nomes).sort_values(ascending=False)


In [ ]:
# Execução das análises EDA e salvamento de arquivos prontos para relatório

# carregar dataset (padrão salvo em outputs/features.csv)
df = carregar_features_csv('outputs/features.csv')
print('Dataset carregado. Shape:', df.shape)

# Escolha de features promissoras (exemplo acadêmico)
#lucas~~ tem que verificar a melhor escolha de promissoras pro nosso caso
promissoras = [
    'V_mean','frac_dark',
    'hist_h_0', 'hist_h_1', 'hist_h_2',
    'perimeter', 'extent',
    'hu_1', 'hu_2',
    'glcm_contrast', 'glcm_energy', 'glcm_correlation'
    'lbp_var'
]
plot_boxplots_por_rotulo(df, promissoras, caminho_saida='outputs/boxplots_promissoras_v2.png')
print('Boxplots salvos: outputs/boxplots_promissoras_v2.png')

# PCA 2D
plot_pca_2d(df, caminho_saida='outputs/pca_2d.png')
print('PCA salvo: outputs/pca_2d.png')

# SelectKBest (top 10)
top_k = selectkbest_features(df, k=10)
print('SelectKBest top:', top_k)

# Importância por RandomForest
imp_rf = importancia_random_forest(df)
imp_rf.head(10).to_csv('outputs/importancia_rf_top10.csv')
print('Importância RF salva em: outputs/importancia_rf_top10.csv')

# Coeficientes da Regressão Logística
imp_lr = coeficientes_regressao_logistica(df)
imp_lr.head(10).to_csv('outputs/coef_logreg_top10.csv')
print('Coeficientes LogReg salvos em: outputs/coef_logreg_top10.csv')

# Salvar tabela final de features selecionadas (consenso simples)
consenso = list(dict.fromkeys(top_k + list(imp_rf.head(10).index) + list(imp_lr.head(10).index)))
pd.DataFrame({'feature': consenso}).to_csv('outputs/features_selecionadas_consenso.csv', index=False)
print('Features selecionadas (consenso) salvas em: outputs/features_selecionadas_consenso.csv')


Dataset carregado. Shape: (979, 13)
Boxplots salvos: boxplots_promissoras_v2.png
PCA salvo: pca_2d.png
SelectKBest top: ['H_mean', 'S_mean', 'V_mean', 'frac_dark', 'area', 'circularity', 'eccentricity', 'glcm_contrast', 'glcm_homogeneity', 'lbp_var']
Importância RF salva em: importancia_rf_top10.csv
Coeficientes LogReg salvos em: coef_logreg_top10.csv
Features selecionadas (consenso) salvas em: features_selecionadas_consenso.csv


c:\Users\cassi\Downloads\A1-VisaoComputacional\Sistema_de_classificacao_de_Imagens\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Extrair features e salvar em outputs/features.csv (pode demorar)
from montar_dataset import montar_dataset
montar_dataset('archive', 'outputs/features.csv', max_por_classe=None)
print('Extração concluída. Arquivo gerado: outputs/features.csv')


Extraindo Withered: 100%|██████████| 54/54 [00:06<00:00,  8.56it/s]

Dataset salvo em features.csv, shape=(979, 13)
Extração concluída.


In [18]:
# Inspecionar colunas de features.csv
import pandas as pd
df_tmp = pd.read_csv('features.csv')
print('Total de features:', len([c for c in df_tmp.columns if c not in ('rotulo','caminho')]))
print('Colunas:', list(df_tmp.columns))
print(df_tmp.head(2).to_dict(orient='records'))


Colunas: ['H_mean', 'S_mean', 'V_mean', 'frac_dark', 'area', 'circularity', 'solidity', 'eccentricity', 'glcm_contrast', 'glcm_homogeneity', 'lbp_var', 'rotulo', 'caminho']
[{'H_mean': 18.493430425835047, 'S_mean': 89.89135891811212, 'V_mean': 154.6376896809064, 'frac_dark': 0.0, 'area': 44219.0, 'circularity': 0.2554182762408096, 'solidity': 0.9253353422478916, 'eccentricity': 0.8022554205186571, 'glcm_contrast': 0.0992907801418439, 'glcm_homogeneity': 0.950354609929078, 'lbp_var': 4.965481706142109, 'rotulo': 'Broken', 'caminho': 'archive\\Broken\\Broken_01.jpg'}, {'H_mean': 21.84431657800456, 'S_mean': 48.34187384648789, 'V_mean': 167.02228313972424, 'frac_dark': 0.0, 'area': 36844.0, 'circularity': 0.3603674291793043, 'solidity': 0.9420367671499068, 'eccentricity': 0.8915762452913231, 'glcm_contrast': 0.0598703724047017, 'glcm_homogeneity': 0.9700648137976492, 'lbp_var': 5.1466620051283005, 'rotulo': 'Broken', 'caminho': 'archive\\Broken\\Broken_02.jpg'}]
